# BNN for MNIST Data Set

By Samuel Stephen and Clayton Wiley

## Data Pre-Processing

In [34]:
# Helper Functions
import numpy as np

sign = np.vectorize(lambda x: float(1) if x > 0 else float(-1))

quantize = np.vectorize(lambda x: float(0) if x == 1 else float(1))

adj = np.vectorize(lambda x: x*2 - 1)

xnor = lambda a, b: 1 if a == b else 0

In [38]:
# Data
import numpy as np
import struct

with open('train-images-idx3-ubyte', 'rb') as f:
    magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
    images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)

with open('train-labels-idx1-ubyte', 'rb') as f:
    magic, num = struct.unpack(">II", f.read(8))
    labels = np.frombuffer(f.read(), dtype=np.uint8)

print(images.shape)
print(labels.shape)


input_img = images[0]
print(labels[0])

(60000, 28, 28)
(60000,)
5


In [ ]:
# Weights
model = np.load('model.npy', allow_pickle=True).item()

fc1w = np.array(model['fc1w'])
fc2w = np.array(model['fc2w'])
fc3w = np.array(model['fc3w'])

print("fc1w shape:", fc1w.shape)
print("fc2w shape:", fc2w.shape)
print("fc3w shape:", fc3w.shape)

def save_as_c_array(filename, array_name, array):
    with open(filename, 'w') as f:
        f.write(f"const {array_name}[{array.shape[0]}][{array.shape[1]}] = {{\n")
        for row in array:
            formatted_row = ", ".join(f"{quantize(sign(val))}" for val in row)
            f.write(f"    {{{formatted_row}}},\n")
        f.write("};\n")

save_as_c_array("fc1w.c", "fc1w", fc1w)
save_as_c_array("fc2w.c", "fc2w", fc2w)
save_as_c_array("fc3w.c", "fc3w", fc3w)

fc1w shape: (128, 784)
fc2w shape: (64, 128)
fc3w shape: (10, 64)


In [36]:
def feed_forward_quantized(input):

    # Layer 1
    X0_input = quantize(adj(sign(input)))
    layer1_output = xnor_matmul(X0_input, fc1w.T)
    layer1_activations = (layer1_output * 2 - cols * rows)

    # Layer 2
    layer2_input = sign(layer1_activations)
    layer2_quantized = quantize(layer2_input)
    layer2_output = xnor_matmul(layer2_quantized, fc2w.T)
    layer2_activations = (layer2_output * 2 - fc1w.shape[0])

    # Layer 3
    layer3_input = sign(layer2_activations)
    layer3_quantized = quantize(layer3_input)
    layer3_output = xnor_matmul(layer3_quantized, fc3w.T)

    final_output = (layer3_output * 2 - fc2w.shape[0])
    A = np.array([final_output], np.int32)

    return A

def xnor_matmul(A, B):
    result = np.zeros((A.shape[0], B.shape[1]), dtype=int)
    for i in range(A.shape[0]):
        for j in range(B.shape[1]):
            xnor_sum = 0
            for k in range(A.shape[1]):
                xnor_sum += xnor(A[i, k], B[k, j])
            result[i, j] = xnor_sum
    return result

In [40]:
out = feed_forward_quantized(input_img)
print(out)
labels[0]

[[[-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -64 -64 -64]
  [-64 -64 -64 -64 -64 -64 -64 -

np.uint8(5)